In [ ]:
#@title ← Run  Mount Google Drive  { display-mode: 'form' }
from google.colab import drive
drive.mount('/content/drive')

import os
import json
from datetime import datetime

# Log hanya di local Colab, bukan Google Drive
os.makedirs('/content/m3u8_logs', exist_ok=True)
LOG_FILE = '/content/m3u8_logs/download_history.json'

if os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'r') as f:
        history = json.load(f)
else:
    history = []

print('✅ Google Drive mounted!')
print(f'📋 Total download history sesi ini: {len(history)} video')
print('='*60)

if len(history) == 0:
    print('📭 Belum ada history download di sesi ini.')
else:
    print('📜 DOWNLOAD HISTORY SESI INI:')
    print('='*60)
    for i, item in enumerate(reversed(history), 1):
        status_icon = '✅' if item['status'] == 'SUCCESS' else '❌'
        print(f"{i}. {status_icon} [{item['date']}] | ⏱️ {item.get('duration','?')}")
        print(f"   📎 M3U8  : {item['m3u8_link']}")
        print(f"   🌐 Ref   : {item['referer']}")
        print(f"   💾 File  : {item['output']}")
        print(f"   📊 Status: {item['status']}")
        print('-'*60)

In [ ]:
#@title ← Run   Paste m3u8 link   { display-mode: 'form' }
import os
import json
import subprocess
import re
from datetime import datetime

M3u8_link = 'https://contoh.com/zz/video.m3u8' #@param {type:'string'}
Referer_link = 'https://contoh.ws/en/zz/' #@param {type:'string'}
Download_location = '/content/drive/MyDrive/myvideo.mp4' #@param {type:'string'}
Cookies_file = '/content/cookies.txt' #@param {type:'string'}

# Log disimpan di local Colab saja
LOG_FILE = '/content/m3u8_logs/download_history.json'
SESSION_LOG = '/content/m3u8_logs/session_log.txt'
os.makedirs('/content/m3u8_logs', exist_ok=True)

print('='*60)
print('🚀 STARTING DOWNLOAD')
print('='*60)
print(f'📎 M3U8    : {M3u8_link}')
print(f'🌐 Referer : {Referer_link}')
print(f'💾 Output  : {Download_location}')
print(f'🍪 Cookies : {Cookies_file}')
print(f'🕐 Time    : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('='*60)

# Install yt-dlp
print('📦 Installing yt-dlp...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'yt-dlp'], capture_output=True)
print('✅ yt-dlp ready!')
print('='*60)
print('⬇️  Download started... (log will appear below)')
print('')

start_time = datetime.now()
log_lines = []

# Jalankan dengan live output
process = subprocess.Popen(
    [
        'yt-dlp',
        '--cookies', Cookies_file,
        '--add-header', f'Referer:{Referer_link}',
        '--add-header', 'User-Agent:Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        '-f', 'bestvideo+bestaudio/best',
        '--merge-output-format', 'mp4',
        '--newline',
        '--progress',
        '-o', Download_location,
        M3u8_link
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

# Baca output line by line (live)
for line in process.stdout:
    line = line.rstrip()
    if not line:
        continue

    log_lines.append(line)

    if '[download]' in line and '%' in line:
        match = re.search(r'(\d+\.?\d*)%.*?([\d.]+\s*\w+/s).*?ETA\s+([\d:]+)', line)
        if match:
            pct = float(match.group(1))
            speed = match.group(2)
            eta = match.group(3)
            bar_filled = int(pct / 5)
            bar = '█' * bar_filled + '░' * (20 - bar_filled)
            print(f'\r⬇️  [{bar}] {pct:.1f}% | 🚀 {speed} | ⏱️ ETA {eta}    ', end='', flush=True)
        else:
            print(f'\r⬇️  {line}    ', end='', flush=True)
    elif '[download] 100%' in line:
        print(f'\r⬇️  [████████████████████] 100% | ✅ Download complete!    ')
    elif '[hlsnative]' in line or '[hls]' in line:
        print(f'🔗 {line}')
    elif '[ffmpeg]' in line or '[merger]' in line:
        print(f'🎬 {line}')
    elif '[generic]' in line or '[info]' in line:
        print(f'ℹ️  {line}')
    elif 'ERROR' in line or 'error' in line.lower():
        print(f'❌ {line}')
    elif 'WARNING' in line:
        print(f'⚠️  {line}')
    else:
        print(f'   {line}')

process.wait()
end_time = datetime.now()
duration = str(end_time - start_time).split('.')[0]
status = 'SUCCESS' if process.returncode == 0 else 'FAILED'
status_icon = '✅' if status == 'SUCCESS' else '❌'

print('')
print('='*60)
print(f'{status_icon} DOWNLOAD {status}!')
print(f'⏱️  Duration : {duration}')
print(f'💾 Saved to : {Download_location}')
print('='*60)

# Simpan history di local Colab
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'r') as f:
        history = json.load(f)
else:
    history = []

history.append({
    'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'm3u8_link': M3u8_link,
    'referer': Referer_link,
    'output': Download_location,
    'status': status,
    'duration': duration
})

with open(LOG_FILE, 'w') as f:
    json.dump(history, f, indent=2)

# Simpan session log di local Colab
with open(SESSION_LOG, 'a') as f:
    f.write(f"\n{'='*60}\n")
    f.write(f"Date     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"M3U8     : {M3u8_link}\n")
    f.write(f"Referer  : {Referer_link}\n")
    f.write(f"Output   : {Download_location}\n")
    f.write(f"Status   : {status}\n")
    f.write(f"Duration : {duration}\n")
    f.write(f"--- Full Log ---\n")
    f.write('\n'.join(log_lines))
    f.write('\n')

print(f'📋 History sesi ini: {len(history)} video')
print('⚠️  Note: History hanya tersimpan selama sesi Colab aktif')
print('')
print('📜 ALL DOWNLOAD HISTORY SESI INI:')
print('='*60)
for i, item in enumerate(reversed(history), 1):
    s_icon = '✅' if item['status'] == 'SUCCESS' else '❌'
    print(f"{i}. {s_icon} [{item['date']}] | ⏱️ {item.get('duration','?')}")
    print(f"   💾 {item['output']}")
    print(f"   📊 {item['status']}")
    print('-'*60)